# CuevanaCinema · 00 Integración PostgreSQL + MongoDB
**Responsable:** Sergio Hernández · Bases de Datos II

Este notebook se conecta **directamente** a las dos bases del proyecto, sin pasar por la API:
- **PostgreSQL** (Neon, `cine_avance1`): lo transaccional — funciones, ventas, staff, contratos.
- **MongoDB** (Atlas, `cine_avance2`): lo flexible — fichas técnicas, requisitos, reseñas, incidentes.

Los dos motores comparten los mismos ids (`id_pelicula`, `id_funcion`, …). El paquete `cuevana/` es el mismo
que usa la API de Postman, así que lo que se ve aquí es exactamente lo que responde la API.

## 0. Preparar el entorno

**En Google Colab:**
1. Abra el icono de la llave 🔑 (*Secretos*) y cree `PG_URL` y `MONGO_URL` con las cadenas de conexión
   (Neon rama `integracion`, base `cine_avance1`, conexión directa; Atlas `ClusterBaseDatos2`).
   Active **Acceso del notebook** en los dos. Las contraseñas nunca se escriben en el notebook (caso C7).
2. Ejecute la celda siguiente: clona el repositorio, instala las dependencias y entra a `integracion-python/`.

> Mientras la integración no esté fusionada en `main`, cambie `RAMA` por la rama del pull request.

**En local:** abra el notebook desde `integracion-python/notebooks/`; usa el archivo `.env`.

In [ ]:
RAMA = "main"   # p. ej. "feature/py-base-integracion" mientras no esté fusionada
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/proyecto-cine-postgresql"):
        !git clone -q -b {RAMA} https://github.com/juanbuga/proyecto-cine-postgresql.git /content/proyecto-cine-postgresql
    %cd /content/proyecto-cine-postgresql/integracion-python
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Carpeta de trabajo:", os.getcwd())

In [ ]:
import pandas as pd
from cuevana import config, db_pg, db_mongo
from cuevana.errores import ErrorCuevana

pd.set_option("display.max_columns", 30, "display.width", 200, "display.max_colwidth", 60)
faltan = [n for n in ("PG_URL", "MONGO_URL") if not config.valor(n)]
assert not faltan, f"Faltan los Secretos/variables: {faltan}"

for motor, ping in (("PostgreSQL", db_pg.ping), ("MongoDB", db_mongo.ping)):
    try:
        info = ping()
        print(f"✔ {motor}: base {info['base']} · {info['version']}")
    except ErrorCuevana as e:
        print(f"✘ {motor}: {e.mensaje}")

In [ ]:
def tabla(filas, **kw):
    """Filas de PostgreSQL o documentos de MongoDB como tabla (los subdocumentos se aplanan)."""
    return pd.json_normalize(filas, **kw) if filas else pd.DataFrame()

def intentar(descripcion, funcion, *args, **kwargs):
    """Ejecuta una operación que la base debe rechazar y muestra el mensaje que devuelve la base."""
    try:
        funcion(*args, **kwargs)
        return {"caso": descripcion, "resultado": "ACEPTADA (revisar)", "estado_http": 200, "mensaje": ""}
    except ErrorCuevana as e:
        d = e.a_dict()["error"]
        regla = "; ".join(d.get("reglas_incumplidas", [])[:2])
        return {"caso": descripcion, "resultado": "RECHAZADA", "estado_http": e.estado,
                "mensaje": (d["mensaje"] + (" → " + regla if regla else ""))[:160]}

In [ ]:
# Limpieza: los datos de prueba se crean y se borran en el mismo notebook (caso C9)
def borrar_funcion(id_funcion):
    for t in ("contratos", "ventas", "funciones"):
        db_pg.consultar(f"DELETE FROM {t} WHERE id_funcion = %s", (id_funcion,))
    try:
        with db_mongo.operacion() as db:
            db.requisitos_proyeccion.delete_many({"id_funcion": id_funcion})
    except ErrorCuevana as e:
        print("Aviso: no se limpiaron los requisitos en MongoDB:", e.mensaje)

import random
from datetime import date, timedelta
def fecha_libre():
    """Fecha aleatoria a +20–60 días a las 11:00: no choca con los datos de prueba ni con otro integrante (C11)."""
    return f"{date.today() + timedelta(days=random.randint(20, 60))} 11:00"

## 1. Una película vista desde los dos motores
PostgreSQL aporta la película y sus funciones; MongoDB, la ficha técnica (reparto, versiones, clasificaciones)
y las reseñas publicadas. Se unen por `id_pelicula`.

In [ ]:
from cuevana import integracion
p = integracion.pelicula(3)
print("PostgreSQL →", p["postgresql"]["pelicula"]["titulo"], "| edad mínima", p["postgresql"]["pelicula"]["edad_minima"])
ficha = p["mongodb"]["ficha_tecnica"]
print("MongoDB    →", ficha["titulo"], "| géneros", ficha["generos"], "| versiones", [v["tipo"] for v in ficha["versiones"]])
print("Reseñas    →", p["mongodb"]["resenas"]["total"], "publicadas, promedio", p["mongodb"]["resenas"]["promedio"])
tabla(p["postgresql"]["funciones"])[["id_funcion", "fecha_hora_inicio", "sede", "sala", "formato", "tipo", "estado"]]

## 2. Una función: ventas (PostgreSQL) + requisitos e incidentes (MongoDB)
`copias_coinciden` compara los datos que MongoDB copia de PostgreSQL (fecha, tipo, versión). PostgreSQL manda.

In [ ]:
especiales = db_mongo.bd().requisitos_proyeccion.distinct("id_funcion")
f = integracion.funcion(especiales[0])
fila = f["postgresql"]["funcion"]
print(f"Función {fila['id_funcion']}: {fila['titulo']} · {fila['sede']} sala {fila['sala']} ({fila['formato']}) · {fila['fecha_hora_inicio']}")
print(f"Boletas pagadas {fila['boletas_pagadas']} de {fila['capacidad']} ({fila['ocupacion_pct']} %) · ingresos ${fila['ingresos']:,.0f}")
req = f["mongodb"]["requisitos_proyeccion"]
print("Experiencias:", req["experiencias"], "· verificación:", req["verificacion"]["estado"])
print("Copias coinciden:", f["copias_coinciden"], f["diferencias"])

## 3. Flujo completo entre motores: programar → vender → cancelar (caso C2)
No hay una transacción que abarque los dos motores. Por eso:
1. Se escribe **primero en PostgreSQL** (el trigger `trg_funciones_validar` decide si la función es válida).
2. Después se escriben los requisitos en MongoDB. **Si MongoDB los rechaza, se borra la función de PostgreSQL** (compensación).
3. Al cancelar, `sp_cancelar_funcion` reembolsa todo en PostgreSQL y luego los requisitos pasan a `NO_APLICA`
   en MongoDB, con reintentos; si MongoDB no responde queda “pendiente” para `sincronizar-copias`.

In [ ]:
from cuevana import procedimientos
REQUISITOS = {"experiencias": ["SUBTITULOS"],
              "requisitos": [{"componente": "SUBTITULOS", "obligatorio": True, "especificacion": {"idioma": "es", "tipo": "SDH"}}],
              "dcp": {"paquete": "COLAB_FTR-1_F_ES-XX_2K", "resolucion": "2K", "cifrado": False}}
nueva = integracion.programar_funcion({"id_sala": random.choice([1, 2, 5, 7, 8]), "id_pelicula": 2,
                                       "fecha_hora_inicio": fecha_libre(), "version": "SUBTITULADA",
                                       "requisitos": REQUISITOS})
ID = nueva["id_funcion"]
print("1) Función", ID, "programada en PostgreSQL; requisitos en MongoDB:", nueva["requisitos"]["verificacion"]["estado"])

venta = procedimientos.registrar_venta(ID, id_espectador=2, id_tipo_boleta=1, cantidad=2, canal="WEB")
print("2) Venta", venta["id_venta"], "por $", venta["total"], "·", venta["avisos"])

cancelada = integracion.cancelar_funcion(ID, "Falla eléctrica en la sede (Colab)")
print("3) PostgreSQL:", cancelada["postgresql"]["avisos"])
print("   MongoDB:", cancelada["mongodb"])
vista = integracion.funcion(ID)
print("4) Estado PG:", vista["postgresql"]["funcion"]["estado"], "· verificación Mongo:",
      vista["mongodb"]["requisitos_proyeccion"]["verificacion"]["estado"], "· copias coinciden:", vista["copias_coinciden"])
borrar_funcion(ID)

In [ ]:
# Compensación: MongoDB rechaza los requisitos (experiencia inexistente) → la función no queda solo en PostgreSQL
antes = db_pg.uno("SELECT COUNT(*) AS n FROM funciones")["n"]
try:
    integracion.programar_funcion({"id_sala": 1, "id_pelicula": 2, "fecha_hora_inicio": fecha_libre(),
                                   "requisitos": {**REQUISITOS, "experiencias": ["HOLOGRAMA"]}})
except ErrorCuevana as e:
    print("Rechazo:", e.mensaje)
    print("Compensación:", e.detalle["compensacion"])
print("Funciones antes", antes, "· después", db_pg.uno("SELECT COUNT(*) AS n FROM funciones")["n"])

## 4. Reseñas validadas contra PostgreSQL (caso C3)
MongoDB no puede revisar llaves foráneas hacia PostgreSQL. Antes de guardar la reseña se comprueba allá que
existan el espectador y la película, y —si es “compra verificada”— que la venta sea de ese espectador, esté
PAGADA y su función ya haya ocurrido.

In [ ]:
pd.DataFrame([
    intentar("Espectador que no existe", integracion.publicar_resena,
             {"id_espectador": 99999, "id_pelicula": 2, "autor_alias": "nadie", "calificacion": 4, "texto": "No debería guardarse."}),
    intentar("Venta de otro espectador", integracion.publicar_resena,
             {"id_espectador": 2, "id_pelicula": 1, "autor_alias": "colab", "calificacion": 5,
              "texto": "Intento usar una venta ajena.", "id_venta": 5}),
    intentar("Calificación 6 ($jsonSchema)", integracion.publicar_resena,
             {"id_espectador": 2, "id_pelicula": 9, "autor_alias": "colab", "calificacion": 6, "texto": "Fuera de rango."}),
])